# Reproduce group GCN: HC3 → GPT‑3.5 Mixed (0% noise), batch 64

This notebook clones the group repository’s `main` branch, builds only HC3 train/validation and GPT‑3.5 Mixed clean graphs, trains the repository’s GCN baseline with **batch size 64**, and evaluates only the clean GPT‑3.5 split. It leaves the cloned repository unchanged apart from generated graph/checkpoint/result files.

In [ ]:
# Kaggle Python 3.13 needs current binary wheels. Do not pin transformers==4.44.2.
!pip install -q --upgrade torch-geometric transformers spacy scikit-learn tqdm
!python -m spacy download en_core_web_sm

import torch
import torch_geometric
import transformers
import spacy
print({
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "transformers": transformers.__version__,
    "spacy": spacy.__version__,
})


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

WORK = Path("/kaggle/working/group_gcn_hc3_to_gpt35_bs64")
REPO = WORK / "PRDetect-Research"
BRANCH = "main"
SEED = 2026
BATCH_SIZE = 64
EPOCHS = 40
RESULT = WORK / "result"
CHECKPOINT = RESULT / f"hc3_gcn_batch{BATCH_SIZE}_seed{SEED}.pth"
TEST_SPLIT = "gpt3.5_mixed_test_split"
if REPO.exists():
    shutil.rmtree(REPO)
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "https://github.com/tphan1420/PRDetect-Research.git", str(REPO)], check=True)
RESULT.mkdir(exist_ok=True)
assert (REPO / "data/original_text/benchmarks/hc3/hc3_train.json").is_file()
assert (REPO / "data/original_text/benchmarks/hc3/hc3_val.json").is_file()
assert (REPO / "data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json").is_file()
print({"commit": subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip(),
       "seed": SEED, "batch_size": BATCH_SIZE, "test": TEST_SPLIT})


In [ ]:
subprocess.run(
    [sys.executable, "building_graph.py", "--file", "hc3_train", "hc3_val", TEST_SPLIT],
    cwd=REPO,
    check=True,
)
for split in ("hc3_train", "hc3_val", TEST_SPLIT):
    assert (REPO / "graph_data" / f"{split}.pkl").is_file()


In [ ]:
subprocess.run(
    [sys.executable, "gcn.py", "--dataset", "hc3", "--train_file", "hc3_train", "--val_file", "hc3_val",
     "--seed", str(SEED), "--batch_size", str(BATCH_SIZE), "--epochs", str(EPOCHS),
     "--lr", "0.0001", "--weight_decay", "0.00001", "--hidden_dim", "256", "--output_dim", "64",
     "--patience", "5", "--save_path", str(CHECKPOINT), "--no_test"],
    cwd=REPO,
    check=True,
)
assert CHECKPOINT.is_file(), f"Missing checkpoint: {CHECKPOINT}"
# Native test.py resolves only model/{dataset}_gcn_model_{seed}.pth.
NATIVE_CHECKPOINT = REPO / "model" / f"hc3_gcn_model_{SEED}.pth"
shutil.copy2(CHECKPOINT, NATIVE_CHECKPOINT)


In [ ]:
# Native project evaluator: its stdout is retained verbatim, and --save appends the native log.
completed = subprocess.run(
    [sys.executable, "test.py", "--dataset", "hc3", "--seed", str(SEED), "--file", TEST_SPLIT,
     "--model_type", "gcn", "--batch_size", str(BATCH_SIZE), "--save"],
    cwd=REPO,
    text=True,
    capture_output=True,
)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
    raise RuntimeError("Native GCN test failed")
(RESULT / "native_test_stdout.txt").write_text(completed.stdout + "\n" + completed.stderr, encoding="utf-8")
native_log = REPO / "result" / "test_result.txt"
assert native_log.is_file()
# The repository tracks historical benchmark lines. Keep only the line that
# this --save invocation just appended, so the result ZIP is unambiguous.
current_log_line = native_log.read_text(encoding="utf-8").strip().splitlines()[-1]
assert TEST_SPLIT in current_log_line and f"seed: {SEED}" in current_log_line and "model: hc3_gcn" in current_log_line
(RESULT / "native_test_result.txt").write_text(current_log_line + "\n", encoding="utf-8")


In [ ]:
import re
import zipfile

stdout = (RESULT / "native_test_stdout.txt").read_text(encoding="utf-8")
match = re.search(r"test_loss:\s*([0-9.]+),\s*test_acc:\s*([0-9.]+),\s*test_f1:\s*([0-9.]+),\s*auc:\s*([0-9.]+)", stdout)
if not match:
    raise RuntimeError("Could not parse the native test output")
loss, accuracy, f1_human, auc = map(float, match.groups())
report = {"model": "group_GCN_main", "seed": SEED, "train_split": "hc3_train", "validation_split": "hc3_val",
          "test_split": TEST_SPLIT, "samples": 1000, "batch_size": BATCH_SIZE, "threshold": 0.5,
          "loss": loss, "accuracy": accuracy, "auc": auc, "f1_human": f1_human,
          "checkpoint": str(CHECKPOINT), "label_convention": {"0": "ai", "1": "human"}}
(RESULT / "gpt35_0pct_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
(RESULT / "run_config.json").write_text(json.dumps({"branch": BRANCH, "seed": SEED, "batch_size": BATCH_SIZE,
    "epochs": EPOCHS, "checkpoint": str(CHECKPOINT), "test": TEST_SPLIT}, indent=2), encoding="utf-8")
archive = WORK / "group_gcn_hc3_to_gpt35_0pct_batch64_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as out:
    for path in sorted(RESULT.iterdir()):
        out.write(path, path.name)
print(json.dumps(report, indent=2))
print("ZIP:", archive)
